# 07 — Curriculum training (manual hyperparameters)

**Summary:** Multi-stage curriculum over the **difficulty-sorted** train/val pool. **Early stopping on `eval_loss` only** per stage. LoRA/optimizer settings come from **`CURRICULUM_BASE_CONFIG`** (you paste from tuning results). Registers one adapter with `curriculum=True`.

**Prerequisites:** Notebook **03** (splits); notebook **02** (or ranked columns) helps difficulty sorting. Set **`CURRICULUM_BASE_CONFIG`** and **`MAX_SEQ_LENGTH`** below after hypertuning.

**Recommended run order:** 03 → 04 → 05 → **08** → 06 → **07** (this notebook) → **09** → 10–12.

**Training subset:** `TRAIN_FIRST_N` caps rows on the **sorted train split** (first N in difficulty order within that split) when set; uses sequential train/val split from the pool when set.

#### Colab / install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install transformers peft accelerate bitsandbytes datasets trl cairosvg pillow scikit-learn lxml pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.5 MB/s eta 0:00:00


#### Imports + parameters

In [3]:
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RUN_PROFILE_ID =  'run_1' #'default'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
EXPERIMENT_ROOT = WORKFLOW_ROOT / 'lora_tuning_workflow'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

SEED = 42
VAL_FRAC = 0.025
TRAIN_FIRST_N = None  # first N rows of sorted train split; None = full train (shuffled split)

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
MAX_SEQ_LENGTH = 2048

CURRICULUM_STAGE_LABELS = ['stage1_easy', 'stage2_easy_med', 'stage3_all']
CURRICULUM_STAGE_FRACS = [0.25, 0.55, 1.0]
CURRICULUM_STAGE_MAX_STEPS = [150, 150, 200]
EVAL_STEPS = 50
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 0.0

# Per-stage step counts override config['max_steps'] during training; keep max_steps >= max(CURRICULUM_STAGE_MAX_STEPS) if you change stages.
CURRICULUM_BASE_CONFIG = {
    'learning_rate': 2e-4,
    'max_steps': max(CURRICULUM_STAGE_MAX_STEPS),
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'lora_r': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.1,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 4,
}

print('WORKFLOW_ROOT:', WORKFLOW_ROOT)
print('cuda:', torch.cuda.is_available())

WORKFLOW_ROOT: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1
cuda: True


#### Build sorted pool + run curriculum

In [4]:
from src.core.dataframe import choose_first_existing, sort_by_difficulty, train_val_split_df
from src.core.modeling_splits import load_train_val_pool
from src.core.runtime import cleanup_memory, set_seed
from src.training.lora.experiments import run_curriculum_experiment_eval_loss_only

set_seed(SEED)
cleanup_memory()

pool = load_train_val_pool(WORKFLOW_ROOT)
ranked_path = PROCESSED_DIR / 'train_ranked.csv'
if ranked_path.exists():
    ranked = pd.read_csv(ranked_path)
    id_set = set(pool['id'].astype(str))
    extra = ranked[ranked['id'].astype(str).isin(id_set)]
    pool = pool.merge(extra[['id'] + [c for c in ranked.columns if c not in pool.columns and c != 'id']], on='id', how='left')
pool = sort_by_difficulty(pool)

PROMPT_COL = choose_first_existing(pool, ['prompt', 'description', 'text'], 'pool')
SVG_COL = choose_first_existing(pool, ['svg', 'svg_code', 'target', 'label'], 'pool')
_shuffle = TRAIN_FIRST_N is None
train_df, val_df = train_val_split_df(
    pool, val_frac=VAL_FRAC, seed=SEED, shuffle=_shuffle
)
train_df = sort_by_difficulty(train_df)
if TRAIN_FIRST_N is not None:
    train_df = train_df.iloc[: int(TRAIN_FIRST_N)].reset_index(drop=True)
    n_val = min(len(val_df), max(1, int(VAL_FRAC * TRAIN_FIRST_N)))
    val_df = val_df.sample(n=n_val, random_state=SEED).reset_index(drop=True)

config = dict(CURRICULUM_BASE_CONFIG)
max_sl = int(MAX_SEQ_LENGTH)

run_name = f'curriculum_seed{SEED}'
summary, gen_df, run_dir, stage_df, reg_id = run_curriculum_experiment_eval_loss_only(
    run_name=run_name,
    config=config,
    reduced_train_df=train_df,
    val_df=val_df,
    prompt_col=PROMPT_COL,
    svg_col=SVG_COL,
    model_id=MODEL_ID,
    max_seq_length=max_sl,
    root_dir=EXPERIMENT_ROOT,
    project_root=PROJECT_DIR,
    stage_fracs=CURRICULUM_STAGE_FRACS,
    stage_labels=CURRICULUM_STAGE_LABELS,
    stage_max_steps=CURRICULUM_STAGE_MAX_STEPS,
    eval_steps=EVAL_STEPS,
    seed=SEED,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    notes='curriculum (manual hparams)',
    models_root=MODELS_ROOT,
)
print('registry_model_id:', reg_id)
display(stage_df)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Adding EOS to eval dataset:   0%|          | 0/1248 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1248 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
50,0.376438,0.349839
100,0.335109,0.336901
150,0.333558,0.327551
200,0.325784,0.326074


registry_model_id: c1a57bce22fe4a00


,stage_idx,stage_label,stage_frac,stage_rows,stage_max_steps,eval_steps,task_metric_name,use_task_metric_early_stopping,early_stopping_patience,early_stopping_min_delta,train_loss,train_runtime,eval_loss,best_loss_checkpoint,stop_reason
0,1,stage1_easy,0.25,12163,150,50,render_rate,False,2,0.0,0.449042,1057.2852,0.382835,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,max_steps reached
1,2,stage2_easy_med,0.55,26759,150,50,render_rate,False,2,0.0,0.386237,1273.2077,0.349035,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,max_steps reached
2,3,stage3_all,1.00,48652,200,50,render_rate,False,2,0.0,0.343529,1802.3313,0.326074,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,max_steps reached
